# Bank Loan Approval Prediction
**Mini project** | Data: Kaggle *Loan Prediction Problem Dataset* | Libraries: `kagglehub`, `pandas`, `seaborn`, `scikit-learn`

Goal: predict whether a loan application is approved (`Y`) or rejected (`N`).

## 1. Install & load data from Kaggle

In [ ]:
!pip install -q kagglehub

In [ ]:
import glob
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("altruistdelhite04/loan-prediction-problem-dataset")

# The dataset ships a train and a test file; only the train file has the Loan_Status label
df = next(d for d in (pd.read_csv(f) for f in glob.glob(f"{path}/*.csv")) if "Loan_Status" in d)
df = df.drop(columns="Loan_ID")

print(df.shape)
df.head()

## 2. Quick exploration

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Missing values:\n", df.isna().sum()[lambda s: s > 0])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="Loan_Status", ax=ax[0]).set_title("Approved vs Rejected")
sns.countplot(data=df, x="Credit_History", hue="Loan_Status", ax=ax[1]).set_title("Credit history vs outcome")
plt.tight_layout()
plt.show()

## 3. Preprocess + train (one scikit-learn pipeline)
Missing values are imputed, categories are one-hot encoded, then a Random Forest is trained.

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Loan_Status"), df["Loan_Status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), selector(dtype_include="number")),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")),
     selector(dtype_exclude="number")),
])

model = make_pipeline(prep, RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
model.fit(X_train, y_train)

## 4. Evaluate

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

pred = model.predict(X_test)
print(classification_report(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.show()

## 5. What drives the decision?

In [ ]:
names = model[:-1].get_feature_names_out()
imp = pd.Series(model[-1].feature_importances_, index=names).nlargest(8)
imp.sort_values().plot.barh(title="Top 8 features")
plt.show()

**Conclusion:** `Credit_History` is typically the strongest predictor, followed by income and loan amount.